# 02 - NeRF or 3D Gaussian Splatting

This notebook uses COLMAP outputs to train a neural renderer and produce images or video.

Paths in this notebook assume you ran notebook 01 and have a sparse COLMAP model.


In [ ]:
from pathlib import Path
import csv
import json
import shutil

ROOT = Path.cwd()
DATASET = "ETs"
CLUSTER = "cluster_0001"
OUT_ROOT = ROOT / "outputs" / f"{DATASET}_{CLUSTER}"
COLMAP_MODEL = OUT_ROOT / "sparse" / "0"

CACHE_ROOT = ROOT / "cache"
meta = json.loads((CACHE_ROOT / DATASET / "meta.json").read_text())
image_ids = meta["image_ids"]
paths = meta["paths"]
id_to_path = dict(zip(image_ids, paths))

cluster_ids = []
with (CACHE_ROOT / "clusters.csv").open("r", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["dataset"] == DATASET and row["scene"] == CLUSTER:
            cluster_ids.append(row["image_id"])

if not cluster_ids:
    raise RuntimeError("Cluster is empty. Check DATASET/CLUSTER and cache/clusters.csv.")

raw_paths = [Path(id_to_path[i]) for i in cluster_ids]
cluster_paths = [p if p.is_absolute() else (ROOT / p) for p in raw_paths]
raw_image_dirs = {p.parent for p in cluster_paths}
if len(raw_image_dirs) != 1:
    raise RuntimeError(f"Expected one image directory, got: {raw_image_dirs}")
RAW_IMAGE_DIR = next(iter(raw_image_dirs))

CLUSTER_IMAGE_DIR = OUT_ROOT / "images"
COPY_IMAGES = True

if COPY_IMAGES:
    CLUSTER_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    for p in cluster_paths:
        dst = CLUSTER_IMAGE_DIR / p.name
        if not dst.exists():
            shutil.copy2(p, dst)

IMAGE_DIR = CLUSTER_IMAGE_DIR if COPY_IMAGES else RAW_IMAGE_DIR

print("COLMAP_MODEL:", COLMAP_MODEL)
print("IMAGE_DIR:", IMAGE_DIR)
print("num images:", len(cluster_paths))


## Option A: Nerfstudio (NeRF)

Steps:
1) Install nerfstudio.
2) Convert COLMAP to nerfstudio format.
3) Train and render.


In [ ]:
# Uncomment to install (requires network)
!pip install nerfstudio

NS_DATA = OUT_ROOT / "nerfstudio_data"
NS_OUTPUT = OUT_ROOT / "nerfstudio_output"

print("NS_DATA:", NS_DATA)
print("NS_OUTPUT:", NS_OUTPUT)


In [ ]:
# Convert COLMAP to nerfstudio data format
# !ns-process-data colmap #   --data {IMAGE_DIR} #   --output-dir {NS_DATA} #   --colmap-model-path {COLMAP_MODEL}


In [ ]:
# Train a NeRF (nerfacto)
# !ns-train nerfacto --data {NS_DATA} --output-dir {NS_OUTPUT}


In [ ]:
# Render a video
# Replace CONFIG with the actual config path from NS_OUTPUT
# CONFIG = NS_OUTPUT / "nerfacto" / "config.yml"
# !ns-render --load-config {CONFIG} --output-path {OUT_ROOT / "nerf_render.mp4"} --rendered-output-names rgb


## Option B: 3D Gaussian Splatting (3DGS)

This assumes you have a 3DGS training repo cloned locally.
Most 3DGS repos expect a COLMAP scene with this structure:
- <scene>/images
- <scene>/sparse/0


In [ ]:
# Prepare a 3DGS scene folder
GS_SCENE = OUT_ROOT / "gaussian_scene"
(GS_SCENE / "images").mkdir(parents=True, exist_ok=True)
(GS_SCENE / "sparse").mkdir(parents=True, exist_ok=True)

# Copy sparse model
if COLMAP_MODEL.exists():
    dst = GS_SCENE / "sparse" / "0"
    dst.mkdir(parents=True, exist_ok=True)
    for item in COLMAP_MODEL.iterdir():
        shutil.copy2(item, dst / item.name)

# Copy images (cluster only)
for p in IMAGE_DIR.glob("*.png"):
    shutil.copy2(p, GS_SCENE / "images" / p.name)

print("GS_SCENE:", GS_SCENE)


In [ ]:
# Example training commands (repo-specific)
# python train.py -s {GS_SCENE} -m {OUT_ROOT / "gaussian_output"}
# python render.py -m {OUT_ROOT / "gaussian_output"} --skip_test --render
